In [3]:
from google_play_scraper import reviews, Sort
import pandas as pd
import sys
import os
import logging

# Add the parent directory to sys.path so that 'src' can be found
sys.path.append(os.path.abspath('../src'))

from preprocess import preprocess_pipeline

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def scrape_app_reviews(app_id, app_name, count=500):
    result, _ = reviews(
        app_id,
        lang='en',
        country='us',
        sort=Sort.NEWEST,
        count=count
    )
    data = []
    for r in result:
        data.append({
            'review': r['content'],
            'rating': r['score'],
            'date': r['at'],
            'bank': app_name,
            'source': 'Google Play'
        })
    return pd.DataFrame(data)

apps = [
    ('com.combanketh.mobilebanking', 'Commercial Bank of Ethiopia'),
    ('com.boa.boaMobileBanking', 'Bank of Abyssinia'),
    ('com.cr2.amolelight', 'Dashen Bank')
]

df_list = []
for app_id, app_name in apps:
    logger.info(f"Scraping {app_name}...")
    try:
        df = scrape_app_reviews(app_id, app_name, count=500)
        df_list.append(df)
        logger.info(f"  -> got {len(df)} reviews")
    except Exception as e:
        logger.error(f"Failed to scrape {app_name}: {e}")

all_reviews = pd.concat(df_list, ignore_index=True)
logger.info(f"Total reviews scraped: {len(all_reviews)}")
all_reviews.head()

2026-05-17 17:49:34,310 - INFO - Scraping Commercial Bank of Ethiopia...
2026-05-17 17:49:36,176 - INFO -   -> got 500 reviews
2026-05-17 17:49:36,181 - INFO - Scraping Bank of Abyssinia...
2026-05-17 17:49:37,747 - INFO -   -> got 500 reviews
2026-05-17 17:49:37,750 - INFO - Scraping Dashen Bank...
2026-05-17 17:49:39,288 - INFO -   -> got 500 reviews
2026-05-17 17:49:39,330 - INFO - Total reviews scraped: 1500


,review,rating,date,bank,source
0,🤙🏼🤙🏼,5,2026-05-16 15:50:50,Commercial Bank of Ethiopia,Google Play
1,worst,1,2026-05-16 12:15:55,Commercial Bank of Ethiopia,Google Play
2,this app very full,5,2026-05-16 09:17:00,Commercial Bank of Ethiopia,Google Play
3,good apps,4,2026-05-16 07:18:33,Commercial Bank of Ethiopia,Google Play
4,ok,5,2026-05-16 03:43:47,Commercial Bank of Ethiopia,Google Play


In [4]:
import os
os.makedirs('data/raw', exist_ok=True)
raw_path = 'data/raw/raw_reviews.csv'
all_reviews.to_csv(raw_path, index=False)
logger.info(f"Raw data saved to {raw_path}")

# Run the reusable preprocessing pipeline
cleaned_path = 'data/raw/cleaned_reviews.csv'
cleaned_df = preprocess_pipeline(raw_path, cleaned_path)

# Verify cleaned data
cleaned_df.head()

2026-05-17 17:49:45,014 - INFO - Raw data saved to data/raw/raw_reviews.csv
2026-05-17 17:49:45,017 - INFO - Starting preprocessing pipeline
2026-05-17 17:49:45,152 - INFO - Loaded 1500 rows from data/raw/raw_reviews.csv
2026-05-17 17:49:45,290 - INFO - Removed 366 duplicate reviews
2026-05-17 17:49:45,301 - INFO - Dropped 0 rows with missing values
2026-05-17 17:49:45,431 - INFO - Date normalisation successful
2026-05-17 17:49:45,435 - INFO - Rating column converted to integer
2026-05-17 17:49:45,495 - INFO - Cleaned data saved to data/raw/cleaned_reviews.csv
2026-05-17 17:49:45,500 - INFO - Preprocessing completed successfully


,review,rating,date,bank,source
0,🤙🏼🤙🏼,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
1,worst,1,2026-05-16,Commercial Bank of Ethiopia,Google Play
2,this app very full,5,2026-05-16,Commercial Bank of Ethiopia,Google Play
3,good apps,4,2026-05-16,Commercial Bank of Ethiopia,Google Play
4,ok,5,2026-05-16,Commercial Bank of Ethiopia,Google Play


In [5]:
print(f"Cleaned shape: {cleaned_df.shape}")
print(cleaned_df['rating'].value_counts().sort_index())

Cleaned shape: (1134, 5)
rating
1    271
2     42
3     80
4    104
5    637
Name: count, dtype: int64
